# Cross-check di Google Colab — Kalibrasi Probabilitas Prediksi Stunting (SKI 2023)

Notebook ini menjalankan ulang seluruh pipeline (`01`–`05`) dan membandingkan hasilnya dengan tabel referensi dari run pertama (`results/reference_comparison_run1.csv`).

**Persiapan:** unggah folder `2026-09-29_calibration_resampling` ke Google Drive (misal `MyDrive/Leakage/project/experiments/`), lalu letakkan `SKI 2023 Balita 0-59 Bulan_asli labels.xlsx` dan `lenanthro.txt` di folder yang sama (atau sesuaikan path di sel 2).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
EXP = '/content/drive/MyDrive/Leakage/project/experiments/2026-09-29_calibration_resampling'  # sesuaikan
XLSX = os.path.join(EXP, 'SKI 2023 Balita 0-59 Bulan_asli labels.xlsx')
LMS  = os.path.join(EXP, 'lenanthro.txt')
os.chdir(EXP); print(os.listdir('.'))

In [ ]:
import sys, sklearn, numpy, pandas, scipy, matplotlib
print(sys.version.split()[0], 'sklearn', sklearn.__version__, 'numpy', numpy.__version__, 'pandas', pandas.__version__)
# Run asli: Python 3.11.15, scikit-learn 1.8.0. Jika versi berbeda, selisih kecil pada angka desimal ke-3/4 wajar.
# Opsional untuk menyamakan versi:
# !pip -q install scikit-learn==1.8.0

In [ ]:
!python 01_prepare_data.py --xlsx "$XLSX" --lms "$LMS"   # ~4 menit (membaca xlsx 64 MB)

In [ ]:
!python 02_train_predict.py --folds 0 1 2 3 4                 # ~20-25 menit (5 fold x 15 model)

In [ ]:
!python 03_evaluate.py      # metrik, bootstrap B=1000, Friedman-Nemenyi
!python 00_flowchart.py
!python 04_figures.py
!python 05_smote_diagnostic.py

## Cross-check angka terhadap run referensi

In [ ]:
import pandas as pd, numpy as np
ref = pd.read_csv('results/reference_comparison_run1.csv')
new = pd.read_csv('results/comparison.csv')
k = ['model','strategy','calibration']
m = ref.merge(new, on=k, suffixes=('_ref','_new'))
cols = [c[:-4] for c in m.columns if c.endswith('_ref')]
diff = pd.DataFrame({c: (m[c+'_new']-m[c+'_ref']).abs() for c in cols})
print('Selisih absolut maksimum per metrik:'); print(diff.max().round(5))
print('\nLULUS (toleransi 0.005)' if (diff.max() < 0.005).all() else '\nPERIKSA: ada selisih > 0.005')

## Tampilkan gambar utama

In [ ]:
from IPython.display import Image, display
for f in sorted(os.listdir('figures')):
    print(f); display(Image(os.path.join('figures', f), width=900))

## Robustness: imbalanced-learn, XGBoost, LightGBM

Menguji (A) kesetaraan ROS/RUS/SMOTE buatan sendiri dengan **imbalanced-learn** pada split yang sama, dan (B) apakah temuan kalibrasi juga berlaku untuk **XGBoost** dan **LightGBM**. Wajib dijalankan setelah `01_prepare_data.py` (butuh `data/analytic.pkl`).

In [ ]:
!pip -q install -U imbalanced-learn xgboost lightgbm
import imblearn, xgboost, lightgbm; print(imblearn.__version__, xgboost.__version__, lightgbm.__version__)

In [ ]:
!python 06_robustness_imblearn_xgboost.py --folds 0            # ~15-30 menit
# versi lengkap (lebih kuat untuk naskah, ~1-2 jam):
# !python 06_robustness_imblearn_xgboost.py --folds 0 1 2 3 4

In [ ]:
import pandas as pd
A = pd.read_csv('results/robustness_imblearn_equivalence.csv')
w = A.pivot_table(index=['fold','model','strategy','calibration'], columns='implementation', values=['AUC','Brier','ECE','Mean_pred'])
d = pd.DataFrame({m:(w[(m,'imblearn')]-w[(m,'own')]).abs() for m in ['AUC','Brier','ECE','Mean_pred']})
print('Selisih maksimum |imblearn - own|:'); print(d.groupby(level='strategy').max().round(4))
print(pd.read_csv('results/robustness_xgb_lgbm_summary.csv').round(4).to_string())
display(Image('figures/fig10_robustness_xgb_lgbm.png', width=900))

Setelah selesai, kirim ke Claude file: `results/robustness_*.csv` dan `results/robustness_versions.json` agar manuskrip (Metode + Tabel suplemen) diperbarui dengan angka nyata.